# Keyhole Melt-Pool Example

This notebook compares MI-DL, SI-DL, and the known Keyhole coordinate using the previously identified dimensionless coordinates. It refits all one-dimensional Gaussian-process models and retains only the fit comparison and summary table.


## 1. Imports and paths

The compact coordinate dataset and the previously identified dimensional formulas are stored in `data/`. Recomputed results are written to `csv/` and the two retained images to `figures/`.


In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

OUTPUT_DIR = Path.cwd()
DATA_DIR = OUTPUT_DIR / "data"
CSV_DIR = OUTPUT_DIR / "csv"
FIG_DIR = OUTPUT_DIR / "figures"
os.environ.setdefault("MPLBACKEND", "Agg")
warnings.filterwarnings("ignore", category=RuntimeWarning)

RANDOM_STATE = 42
TEST_SIZE = 0.25
METHODS = (
    ("MI-DL", "MI-DL_pi", r"MI-DL $\Pi_1$"),
    ("SI-DL", "SI-DL_pi", r"SI-DL $\Pi_1$"),
    ("Known Ke", "Known Ke_pi", r"Known Ke $\Pi_1$"),
)
CSV_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load the coordinates

The response is the normalized keyhole quantity $e^*$. All three methods use the same deterministic train/test split.


In [ ]:
coordinates = pd.read_csv(DATA_DIR / "coordinates.csv")
reference_summary = pd.read_csv(DATA_DIR / "reference_summary.csv")
y = coordinates["e_star"].to_numpy(float)
train_idx, test_idx = train_test_split(
    np.arange(y.size),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
coordinates.head()


## 3. Refit the GPR models

Fit one standardized GPR model for each raw dimensionless coordinate and recompute held-out normalized errors.


In [ ]:
fit_records = []
models = {}

for method, column, _ in METHODS:
    feature = coordinates[column].to_numpy(float).reshape(-1, 1)
    kernel = ConstantKernel(1.0, (1.0e-3, 1.0e3)) * RBF(1.0, (1.0e-3, 1.0e3))
    kernel += WhiteKernel(1.0, (1.0e-8, 1.0e3))
    model = make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel,
            alpha=1.0e-8,
            normalize_y=True,
            n_restarts_optimizer=8,
            random_state=RANDOM_STATE,
        ),
    )
    model.fit(feature[train_idx], y[train_idx])
    prediction = model.predict(feature[test_idx])
    mse_raw = float(mean_squared_error(y[test_idx], prediction))
    error_scale = float(np.var(y[train_idx], ddof=0))
    models[method] = model
    fit_records.append({
        "method": method,
        "gpr_mse_raw": mse_raw,
        "gpr_mse_normalized": mse_raw / error_scale,
        "gpr_rmse_normalized": np.sqrt(mse_raw / error_scale),
        "n_train": train_idx.size,
        "n_test": test_idx.size,
    })

gpr_summary = pd.DataFrame(fit_records)
gpr_summary.to_csv(CSV_DIR / "gpr_summary.csv", index=False)
gpr_summary


## 4. Save the compact method summary

Combine the identified formulas and information scores with the newly computed raw-coordinate GPR errors.


In [ ]:
summary = reference_summary.copy()
summary = summary.drop(
    columns=[
        "gpr_mse_normalized_raw_pi",
        "gpr_rmse_normalized_raw_pi",
        "rank_by_gpr_raw_pi",
    ],
    errors="ignore",
)
summary = summary.merge(gpr_summary, on="method", how="left")
summary["rank_by_gpr"] = summary["gpr_mse_normalized"].rank(method="min").astype(int)
summary.to_csv(CSV_DIR / "summary.csv", index=False)
summary


## 5. GPR fit comparison

Show the data and fitted GPR mean for MI-DL, SI-DL, and the known Keyhole coordinate.


In [ ]:
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "axes.titlesize": 22,
    "axes.labelsize": 21,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "legend.fontsize": 15,
})
fig, axes = plt.subplots(1, 3, figsize=(17.2, 5.7), sharey=True, dpi=260)
for ax, (method, column, xlabel) in zip(axes, METHODS):
    feature = coordinates[column].to_numpy(float).reshape(-1, 1)
    line_x = np.linspace(float(feature.min()), float(feature.max()), 320).reshape(-1, 1)
    line_y = models[method].predict(line_x)
    mse = float(gpr_summary.loc[gpr_summary["method"].eq(method), "gpr_mse_normalized"].iloc[0])
    ax.scatter(feature[:, 0], y, s=42, color="#1f77b4", alpha=0.58, edgecolors="white", linewidths=0.45, label="data")
    ax.plot(line_x[:, 0], line_y, color="#d62728", linewidth=2.9, label="GPR mean")
    ax.text(0.97, 0.06, f"GPR norm. MSE = {mse:.4f}", transform=ax.transAxes, ha="right", va="bottom", fontsize=15, bbox={"boxstyle":"round,pad=0.24","facecolor":"white","edgecolor":"#d1d5db","alpha":0.92})
    ax.set_xlabel(xlabel)
    ax.set_title(method)
    ax.grid(True, alpha=0.22, linewidth=0.9)
    ax.legend(frameon=True, facecolor="white", edgecolor="#d1d5db", framealpha=0.94, loc="upper left")
axes[0].set_ylabel(r"$e^*$")
fig.suptitle("Keyhole", fontsize=25, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "fit.png", bbox_inches="tight", facecolor="white")
plt.close(fig)


## 6. Summary table

Render the compact comparison table as the second and final retained image.


In [ ]:
view_columns = [
    "method", "formula", "mutual_information", "epsilon_lb_normalized",
    "S_cov", "gpr_mse_normalized", "rank_by_MI", "rank_by_gpr",
]
view = summary[view_columns].copy()
for column in view.columns:
    if pd.api.types.is_float_dtype(view[column]):
        view[column] = view[column].map(lambda value: f"{value:.6g}")
fig, ax = plt.subplots(figsize=(16.0, 3.4), dpi=220)
ax.axis("off")
ax.set_title("Keyhole summary", fontsize=18, weight="bold", pad=14)
table = ax.table(cellText=view.values, colLabels=view.columns, cellLoc="center", colLoc="center", bbox=[0.02,0.06,0.96,0.78])
table.auto_set_font_size(False)
for (row, _), cell in table.get_celld().items():
    cell.set_edgecolor("#3f3f46")
    cell.set_linewidth(0.75)
    cell.PAD = 0.03
    if row == 0:
        cell.set_facecolor("#1f2937")
        cell.get_text().set_color("white")
        cell.get_text().set_weight("bold")
        cell.get_text().set_fontsize(8.8)
    else:
        cell.set_facecolor("#f8fafc" if row % 2 else "#ffffff")
        cell.get_text().set_fontsize(8.0)
fig.savefig(FIG_DIR / "summary.png", bbox_inches="tight", facecolor="white")
plt.close(fig)
